# **Interactive exploration of Antarctic freshwater fluxes by Zwally drainage basin**

*Time-series and map-based comparison of SSP126 and SSP585 from 1990 to 2300*




For an interactive version of this page please visit the Google Colab: [Open in Google Colab](https://colab.research.google.com/drive/1XLlTaxzELRH4_yD6Tcj8xCvUEbQyHeFP)

*(To open link in new tab press Ctrl + click)*

Alternatevly this notebook can be opened with Binder by following the link: [Interactive exploration of Antarctic freshwater fluxes by Zwally drainage basin](https://mybinder.org/v2/gh/s4oceanice/literacy.s4oceanice/main?urlpath=%2Fdoc%2Ftree%2Fnotebooks_binder%2Foceanice_SSP126vsSSP585_ZwallyBasins.ipynb)

**Scientific background**

The Antarctic Ice Sheet (AIS) is one of the largest sources of uncertainty in future projections of sea level rise and Southern Ocean circulation. The freshwater it releases into the Southern Ocean — through surface melt, sub-shelf melting of ice shelves, and iceberg calving — affects deep water formation and ocean stratification.


## **Notebook objectives**

This notebook enables users to:

- compare SSP126 and SSP585 by drainage basin;
- explore four freshwater-flux components;
- select a basin and time interval interactively;
- reconstruct basin geometries from boundary coordinates;
- map projected values for a selected year and scenario.

**Data source**

The data, available on the OCEAN ICE ERDDAP server and contain projections spanning **1990–2300** for the **27 Zwally glaciological drainage basins**.

Two dataset are available:
- **Low-emission scenario** (Scenario SSP126): https://er1.s4oceanice.eu/erddap/griddap/SSP126_FWF_1990_2300_ZwallyBasins
- **High-emission scenario** (Scenario SSP585): https://er1.s4oceanice.eu/erddap/griddap/SSP585_FWF_1990_2300_ZwallyBasins

The datasets include four freshwater-flux components:

- net mass balance (*Net_Mass_Balance*),
- surface meltwater runoff (*Surface_Meltwater_Runoff_Fluxes*),
- sub-shelf melt of ice shelves (*Sub_Shelf_Melt_Fluxes*),
- and calving flux (*Calving_Fluxes*).

For each variable, projections are available for the 0.05, 0.50 and 0.95 quantiles, allowing both the median projection and its associated uncertainty interval to be visualized.

**How to use this notebook**

1. Run the code cells sequentially from top to bottom.
2. Wait for the remote dataset to be downloaded and processed.
3. Use the available menus, sliders, or map controls to select the variables and periods of interest.
4. Read the interpretation guidance before drawing scientific conclusions from the visualizations.

The notebook retrieves data from remote services. An active internet connection is therefore required.


**1. Data retrieval and preprocessing**

The following code downloads the two scenario datasets, removes the ERDDAP units row, converts the time, basin, quantile, and freshwater fields to numerical values, and prepares the source tables for interactive filtering.

In [ ]:
# @title
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

url_low = "https://er1.s4oceanice.eu/erddap/griddap/SSP126_FWF_1990_2300_ZwallyBasins.csv?Net_Mass_Balance%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(27.0)%5D,Surface_Meltwater_Runoff_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(27.0)%5D,Sub_Shelf_Melt_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(27.0)%5D,Calving_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(27.0)%5D"

url_high = "https://er1.s4oceanice.eu/erddap/griddap/SSP585_FWF_1990_2300_ZwallyBasins.csv?Net_Mass_Balance%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(27.0)%5D,Surface_Meltwater_Runoff_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(27.0)%5D,Sub_Shelf_Melt_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(27.0)%5D,Calving_Fluxes%5B(0.05):1:(0.95)%5D%5B(1990.0):1:(2300.0)%5D%5B(1.0):1:(27.0)%5D"


def load_data(url):
    df = pd.read_csv(url, skiprows=[1])
    df.columns = [c.strip() for c in df.columns]

    if "time" not in df.columns:
        time_cols = [c for c in df.columns if "time" in c.lower()]
        if time_cols:
            df = df.rename(columns={time_cols[0]:"time"})

    df["time"] = pd.to_numeric(df["time"], errors="coerce").astype("Int64")
    df["basin"] = pd.to_numeric(df["basin"], errors="coerce").astype("Int64")
    df["quantile"] = pd.to_numeric(df["quantile"], errors="coerce")

    return df


df_low = load_data(url_low)
df_high = load_data(url_high)


**2. Interactive controls**

This section defines the basin, variable, and time-range selectors, together with the output area used by the time-series visualization.

In [ ]:
# @title
basin_map = {f"Basin {i}": i for i in range(1, 28)}

columns_to_plot = [
    "Net_Mass_Balance",
    "Surface_Meltwater_Runoff_Fluxes",
    "Sub_Shelf_Melt_Fluxes",
    "Calving_Fluxes"
]

basin_dropdown = widgets.Dropdown(
    options=basin_map,
    value=1,
    description="Basin:"
)

variable_dropdown = widgets.Dropdown(
    options=columns_to_plot,
    value=columns_to_plot[0],
    description="Variable:"
)

time_slider = widgets.IntRangeSlider(
    value=[1990, 2300],
    min=1990,
    max=2300,
    step=1,
    description="Years:",
    layout={"width": "500px"}
)

output = widgets.Output()

**3. Basin-level scenario comparison**

The update function filters both scenarios for the selected basin and period, extracts the median and uncertainty quantiles, and redraws the comparison whenever a control changes.


In [ ]:
# @title
def update_plot(change=None):
    with output:
        clear_output(wait=True)

        bid = basin_dropdown.value
        var = variable_dropdown.value
        tmin, tmax = time_slider.value
        basin_name = [name for name, value in basin_map.items() if value == bid][0]

        def get_plot_elements(df):
            filtered = df[
                (df["basin"] == bid)
                & (df["time"] >= tmin)
                & (df["time"] <= tmax)
            ].copy()

            median = filtered[filtered["quantile"] == 0.50].sort_values("time")
            lower = filtered.groupby("time")[var].min().sort_index()
            upper = filtered.groupby("time")[var].max().sort_index()

            return median, lower, upper

        m_low, l_low, u_low = get_plot_elements(df_low)
        m_high, l_high, u_high = get_plot_elements(df_high)

        plt.figure(figsize=(12, 7))

        plt.fill_between(
            l_low.index,
            l_low.values,
            u_low.values,
            color="blue",
            alpha=0.15,
            label="Low Emissions (SSP126) Range"
        )

        plt.plot(
            m_low["time"],
            m_low[var],
            color="blue",
            linewidth=2,
            label="Low Emissions (SSP126) Median"
        )

        plt.fill_between(
            l_high.index,
            l_high.values,
            u_high.values,
            color="red",
            alpha=0.15,
            label="High Emissions (SSP585) Range"
        )

        plt.plot(
            m_high["time"],
            m_high[var],
            color="red",
            linewidth=2,
            label="High Emissions (SSP585) Median"
        )

        plt.title(f"{var} - {basin_name}")
        plt.xlabel("Year")
        plt.ylabel("Gt/yr")
        plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()


basin_dropdown.observe(update_plot, names="value")
variable_dropdown.observe(update_plot, names="value")
time_slider.observe(update_plot, names="value")

display(widgets.VBox([
    basin_dropdown,
    variable_dropdown,
    time_slider,
    output
]))

update_plot()

**4. Reconstruction of Zwally basin geometries**

The next section downloads the basin-boundary coordinates, converts them to `EPSG:3031`, reconstructs polygon and multipolygon geometries, validates the results, and defines a repair procedure for Basin 2.


In [ ]:
# @title
import requests
import pandas as pd
import geopandas as gpd

from pyproj import Transformer
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
from shapely.validation import make_valid


coords_url = "https://earth.gsfc.nasa.gov/sites/default/files/lab_cryo/data/polar_ice_altimetry/antarctic_and_greenland_drainage_systems/ant_full_drainagesystem_polygons.txt"

response = requests.get(coords_url)
response.raise_for_status()

lines = response.text.splitlines()

coords_dict = {}

for line in lines:
    parts = line.split()

    if len(parts) == 3:
        try:
            lat = float(parts[0])
            lon = float(parts[1])
            bid = int(parts[2])

            if bid not in coords_dict:
                coords_dict[bid] = []

            coords_dict[bid].append((lon, lat))

        except ValueError:
            continue


transformer = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)


def clean_consecutive_duplicates(pts):
    if not pts:
        return pts

    clean_pts = [pts[0]]

    for p in pts[1:]:
        if p != clean_pts[-1]:
            clean_pts.append(p)

    if len(clean_pts) > 1 and clean_pts[0] == clean_pts[-1]:
        clean_pts = clean_pts[:-1]

    return clean_pts


def iter_polygons(geom):
    if geom is None or geom.is_empty:
        return

    if geom.geom_type == "Polygon":
        yield geom

    elif geom.geom_type == "MultiPolygon":
        for part in geom.geoms:
            yield part

    elif geom.geom_type == "GeometryCollection":
        for part in geom.geoms:
            yield from iter_polygons(part)


def clean_geometry(geom):
    if geom is None or geom.is_empty:
        return None

    try:
        geom = make_valid(geom)
    except Exception:
        geom = geom.buffer(0)

    try:
        geom = geom.buffer(0)
    except Exception:
        pass

    polygons = [p for p in iter_polygons(geom) if p is not None and not p.is_empty]

    if not polygons:
        return None

    if len(polygons) == 1:
        cleaned = polygons[0]
    else:
        cleaned = MultiPolygon(polygons)

    try:
        cleaned = make_valid(cleaned)
    except Exception:
        cleaned = cleaned.buffer(0)

    if cleaned.is_empty:
        return None

    return cleaned


def build_geometry_from_points(pts):
    pts = clean_consecutive_duplicates(pts)

    if len(pts) < 3:
        return None

    projected_pts = []

    for lon, lat in pts:
        x, y = transformer.transform(lon, lat)
        projected_pts.append((x, y))

    projected_pts = clean_consecutive_duplicates(projected_pts)

    if len(projected_pts) < 3:
        return None

    geom = Polygon(projected_pts)
    geom = clean_geometry(geom)

    return geom


def safe_unary_union(geometries):
    clean_geometries = []

    for geom in geometries:
        cleaned = clean_geometry(geom)

        if cleaned is not None and not cleaned.is_empty:
            clean_geometries.append(cleaned)

    if not clean_geometries:
        return None

    try:
        return unary_union(clean_geometries)
    except Exception:
        buffered = []

        for geom in clean_geometries:
            try:
                buffered.append(geom.buffer(0))
            except Exception:
                continue

        return unary_union(buffered)


def repair_basin_2_from_internal_gap(
    gdf,
    basin_col="basin",
    basin_id=2,
    min_gap_area_km2=1000,
    max_gap_distance_m=300000
):
    gdf_work = gdf.copy()

    gdf_work["geometry"] = gdf_work["geometry"].apply(clean_geometry)
    gdf_work = gdf_work[gdf_work["geometry"].notna()].copy()

    union_geom = safe_unary_union(gdf_work.geometry.tolist())

    if union_geom is None or union_geom.is_empty:
        print("Geometry union failed. Basin 2 not modified.")
        return gdf

    internal_gaps = []

    for poly in iter_polygons(union_geom):
        for ring in poly.interiors:
            gap = Polygon(ring)
            gap = clean_geometry(gap)

            if gap is None or gap.is_empty:
                continue

            gap_area_km2 = gap.area / 1_000_000

            if gap_area_km2 >= min_gap_area_km2:
                internal_gaps.append(gap)

    if not internal_gaps:
        print("No significant internal holes found. Basin 2 not modified.")
        return gdf_work

    basin_mask = gdf_work[basin_col] == basin_id

    if not basin_mask.any():
        print(f"Basin {basin_id} not found. Basin 2 not modified.")
        return gdf_work

    basin_geom = gdf_work.loc[basin_mask, "geometry"].iloc[0]

    nearby_gaps = [
        gap
        for gap in internal_gaps
        if gap.distance(basin_geom) <= max_gap_distance_m
    ]

    if nearby_gaps:
        gaps_to_add = nearby_gaps
    else:
        gaps_to_add = [min(internal_gaps, key=lambda gap: gap.distance(basin_geom))]

    repaired_geom = safe_unary_union([basin_geom] + gaps_to_add)
    repaired_geom = clean_geometry(repaired_geom)

    if repaired_geom is None or repaired_geom.is_empty:
        print(f"Basin {basin_id} repair failed. Original geometry preserved.")
        return gdf_work

    gdf_work.loc[basin_mask, "geometry"] = repaired_geom

    print(f"Basin {basin_id} repaired by adding {len(gaps_to_add)} internal hole(s).")

    return gdf_work


**5. Assembly and validation of the basin layer**

The reconstructed geometries are assembled into a `GeoDataFrame`, and the Basin 2 repair routine is applied before spatial visualization.





In [ ]:
# @title
basins_list = []

for bid, pts in coords_dict.items():
    geom = build_geometry_from_points(pts)

    if geom is None:
        continue

    basins_list.append({
        "basin": bid,
        "geometry": geom
    })

df_geom = pd.DataFrame(basins_list)

gdf_basins = gpd.GeoDataFrame(
    df_geom,
    geometry="geometry",
    crs="EPSG:3031"
)

gdf_basins = repair_basin_2_from_internal_gap(gdf_basins)

print("Geometries reconstructed in EPSG:3031, and Basin 2 successfully repaired.")

No significant internal holes found. Basin 2 not modified.
Geometries reconstructed in EPSG:3031, and Basin 2 successfully repaired.


**6. Interactive map controls**

The following cell defines the year, variable, scenario, and selected-basin controls used by the map and prepares the scenario data for spatial joining.

In [ ]:
# @title
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import geopandas as gpd
import pandas as pd
import numpy as np


map_year_slider = widgets.IntSlider(
    value=2023,
    min=1990,
    max=2300,
    description="Year:",
    continuous_update=True,
    layout={"width": "500px"}
)

map_variable = widgets.Dropdown(
    options=columns_to_plot,
    value="Net_Mass_Balance",
    description="Variabile:"
)

map_scenario = widgets.Dropdown(
    options=[
        ("SSP126 (Low)", "low"),
        ("SSP585 (High)", "high")
    ],
    value="low",
    description="Scenario:"
)

selected_basin_dropdown = widgets.Dropdown(
    options=[(f"Basin {i}", i) for i in range(1, 28)],
    value=1,
    description="Basin:"
)

map_output = widgets.Output()


def get_active_year_data():
    year = map_year_slider.value
    var = map_variable.value
    scenario = map_scenario.value

    df_active = df_low if scenario == "low" else df_high

    year_data = df_active[
        (df_active["time"] == year)
        & (df_active["quantile"] == 0.5)
    ][["basin", var]].copy()

    year_data[var] = pd.to_numeric(year_data[var], errors="coerce")

    return year_data


def get_selected_basin_value(year_data):
    var = map_variable.value
    basin_id = selected_basin_dropdown.value

    row = year_data[year_data["basin"] == basin_id]

    if row.empty:
        return "N/D"

    value = row[var].iloc[0]

    if pd.isna(value):
        return "N/D"

    return f"{value:.6f}"

**7. Spatial visualization**

The final code section joins projected values to the basin geometries, draws the Antarctic basin map, highlights the selected basin, and displays its value in an information box.


**Interactive visualizations**

The notebook contains two complementary views:

1. a time-series comparison for one selected basin;
2. a map of all basins for one selected year, variable, and scenario.

The line plots show median values and uncertainty intervals. The map supports comparison of the spatial distribution and highlights the basin chosen in the selector.


In [ ]:
# @title
def draw_detail_box_inside_map(ax, year_data):
    year = map_year_slider.value
    var = map_variable.value
    scenario = map_scenario.value
    basin_id = selected_basin_dropdown.value
    value_display = get_selected_basin_value(year_data)

    detail_text = (
        "Selected Basin\n"
        f"Basin: {basin_id}\n"
        f"Variable: {var}\n"
        f"Value: {value_display} Gt/yr"
    )

    ax.text(
        0.02,
        0.98,
        detail_text,
        transform=ax.transAxes,
        fontsize=10,
        va="top",
        ha="left",
        bbox={
            "boxstyle": "round,pad=0.5",
            "facecolor": "white",
            "edgecolor": "black",
            "alpha": 0.85
        }
    )


def draw_map(change=None):
    year = map_year_slider.value
    var = map_variable.value
    scenario = map_scenario.value
    selected_basin = selected_basin_dropdown.value

    with map_output:
        clear_output(wait=True)

        year_data = get_active_year_data()

        map_gdf = gdf_basins.merge(
            year_data,
            on="basin",
            how="left"
        )

        if map_gdf.crs is None:
            map_gdf = map_gdf.set_crs("EPSG:3031")

        if map_gdf.crs.to_string() != "EPSG:3031":
            map_gdf_polar = map_gdf.to_crs("EPSG:3031")
        else:
            map_gdf_polar = map_gdf.copy()

        fig, ax = plt.subplots(figsize=(12, 10))

        cmap = "RdBu" if "Mass_Balance" in var else "YlOrRd"

        map_gdf_polar.plot(
            column=var,
            ax=ax,
            cmap=cmap,
            legend=True,
            legend_kwds={
                "label": f"{var} (Gt/yr)",
                "orientation": "vertical",
                "shrink": 0.5
            },
            edgecolor="black",
            linewidth=0.6,
            missing_kwds={
                "color": "lightgrey",
                "edgecolor": "black",
                "hatch": "///",
                "label": "Missing values"
            }
        )

        selected_gdf = map_gdf_polar[map_gdf_polar["basin"] == selected_basin]

        if not selected_gdf.empty:
            selected_gdf.boundary.plot(
                ax=ax,
                color="yellow",
                linewidth=3
            )

        for _, row in map_gdf_polar.iterrows():
            if row.geometry is None or row.geometry.is_empty:
                continue

            point = row.geometry.representative_point()

            ax.annotate(
                text=str(int(row["basin"])),
                xy=(point.x, point.y),
                color="black",
                fontsize=9,
                fontweight="bold",
                ha="center",
                va="center",
                bbox={
                    "boxstyle": "circle,pad=0.3",
                    "fc": "white",
                    "alpha": 0.75,
                    "ec": "none"
                }
            )

        draw_detail_box_inside_map(ax, year_data)

        ax.set_title(
            f"Year {year} - {var} ({scenario.upper()})",
            fontsize=14
        )

        ax.axis("off")
        plt.show()


map_scenario.observe(draw_map, names="value")
map_variable.observe(draw_map, names="value")
map_year_slider.observe(draw_map, names="value")
selected_basin_dropdown.observe(draw_map, names="value")


display(
    widgets.VBox([
        widgets.HTML("<h4>Visualizzatore Antartico Ibrido</h4>"),
        widgets.HBox([
            map_scenario,
            map_variable,
            map_year_slider
        ]),
        widgets.HBox([
            selected_basin_dropdown
        ]),
        map_output
    ])
)

draw_map()

**Interpretation guidance**

Basin-level projections reveal regional heterogeneity but depend on the adopted drainage-basin boundaries. Values near zero, missing records, or abrupt changes should be checked against the source tables and quantile availability.

The reconstructed geometries originate from an external boundary file. Geometry repair is applied specifically to Basin 2, and the resulting spatial layer should be visually validated before publication or quantitative geospatial analysis.


## Additional Resources

The Python libraries used in this notebook are:

- [pandas](https://pandas.pydata.org/docs/)
- [matplotlib](https://matplotlib.org/stable/index.html)
- [ipywidgets](https://ipywidgets.readthedocs.io/)
- [GeoPandas](https://geopandas.org/en/stable/)
- [Shapely](https://shapely.readthedocs.io/en/stable/)
- [request](https://realpython.com/python-requests/)

This work has received funding from the European Union Horizon Europe project Ocean-Cryosphere Exchanges in ANtarctica: Impacts on Climate and the Earth System (OCEAN ICE) under grant agreement No. 101060452 (<https://doi.org/10.3030/101060452>). UK partners are funded by UK Research and Innovation (UKRI) under the UK government's Horizon Europe funding guarantee.

The freshwater-flux datasets are distributed through the OCEAN:ICE ERDDAP infrastructure.


<center>
  <div style="display: flex; justify-content: center; align-items: flex-start; gap: 80px;">
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/TO-USE-RGB-for-digital-materials-V.png" height="140" style="margin-top: 50px;"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/UKRI-logo-1.png" height="100"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2023/06/logo-polar-cluster-2.png" height="100"/>
  </div>
</center>